# System Tests

In [ ]:
import random
import jax
import jax.numpy as jnp

In [ ]:
from utils import batch_mul
import samplers
from sde import VPSDE, subVPSDE, VESDE

### General Variables

In [ ]:
batch_size = 4
normal = jax.random.normal
jax.random.normal = lambda key, shape: jnp.full(shape, 0.5)

## Utility Functions

### Batch Multiplications

In [ ]:
a = jnp.array([1.0,2.0, 0.5, 10.0]) # (4,)
b = jnp.ones((batch_size, 32,32,3)) # (4,32,32,3)
result = batch_mul(a,b)

assert result.shape == b.shape, f'Expected {b.shape} array, got {result.shape}'

expected = a.reshape(batch_size, 1,1,1)

assert jnp.allclose(result, expected), "Result does not match expected product"

## SDEs

In [ ]:
shape = (batch_size,4,4,3)
x = jnp.ones(shape)
t = jnp.full((batch_size,),0.5)
score = jnp.full(shape, 0.7)

### VP SDE

In [ ]:
vp_sde = VPSDE(0.1, 20, 10)
rng = jax.random.PRNGKey(0)
mean, std = vp_sde.marginal_prob(x,t)
f, g = vp_sde.sde(x,t)
prior_sample = vp_sde.prior_sampling(rng, shape)
rev_f, rev_g = vp_sde.reverse_sde(x,t,score,False)
rev_f_flow, _ = vp_sde.reverse_sde(x,t,score,True)

expected_f = jnp.full(shape, -5.025)
expected_g = jnp.full((batch_size,), 3.17017)
expected_mean = jnp.full(shape, 0.281184)
expected_std = jnp.full(shape, 0.959654)
expected_sample = jnp.full(shape, 0.5)
expected_rev_f = jnp.full(shape, -12.06)
expected_rev_g = jnp.full((batch_size,),3.17017)
expected_rev_f_flow = jnp.full(shape, -8.5425)


assert jnp.allclose(f, expected_f), 'Mismatch in f'
assert jnp.allclose(g, expected_g), 'Mismatch in g'
assert jnp.allclose(mean, expected_mean), 'Mismatch in mean'
assert jnp.allclose(std, expected_std), 'Mismatch in std'
assert jnp.allclose(prior_sample, expected_sample), 'Mismatch in prior sample'
assert jnp.allclose(rev_f, expected_rev_f), 'Mismatch in rev_f'
assert jnp.allclose(rev_g, expected_rev_g), 'Mismatch in rev_g'
assert jnp.allclose(rev_f_flow, expected_rev_f_flow), 'Mismatch in rev_f when probability flow=True'

### Sub-VP SDE

In [ ]:
sub_vp_sde = subVPSDE(0.1, 20, 10)
rng = jax.random.PRNGKey(0)
mean, std = sub_vp_sde.marginal_prob(x,t)
f, g = sub_vp_sde.sde(x,t)
prior_sample = sub_vp_sde.prior_sampling(rng, shape)
rev_f, rev_g = sub_vp_sde.reverse_sde(x,t,score,False)
rev_f_flow, _ = sub_vp_sde.reverse_sde(x,t,score,True)

expected_f = jnp.full(shape, -5.025)
expected_g = jnp.full((batch_size,), 3.16024)
expected_mean = jnp.full(shape, 0.281184)
expected_std = jnp.full(shape, 0.92093)
expected_sample = jnp.full(shape, 0.5)
expected_rev_f = jnp.full(shape, -12.016024)
expected_rev_g = jnp.full((batch_size,), 3.16024)
expected_rev_f_flow = jnp.full(shape, -8.5205)


assert jnp.allclose(f, expected_f), 'Mismatch in f'
assert jnp.allclose(g, expected_g), 'Mismatch in g'
assert jnp.allclose(mean, expected_mean), 'Mismatch in mean'
assert jnp.allclose(std, expected_std), 'Mismatch in std'
assert jnp.allclose(prior_sample, expected_sample), 'Mismatch in prior sample'
assert jnp.allclose(rev_f, expected_rev_f), 'Mismatch in rev_f'
assert jnp.allclose(rev_g, expected_rev_g), 'Mismatch in rev_g'
assert jnp.allclose(rev_f_flow, expected_rev_f_flow), 'Mismatch in rev_f when probability flow=True'

### VE SDE

In [ ]:
ve_sde = VESDE(0.01, 50, 10)
rng = jax.random.PRNGKey(0)
mean, std = ve_sde.marginal_prob(x,t)
f, g = ve_sde.sde(x,t)
prior_sample = ve_sde.prior_sampling(rng, shape)
rev_f, rev_g = ve_sde.reverse_sde(x,t,score,False)
rev_f_flow, _ = ve_sde.reverse_sde(x,t,score,True)

expected_f = jnp.full(shape, 0)
expected_g = jnp.full((batch_size,), 2.918423)
expected_mean = jnp.full(shape, 1.0)
expected_std = jnp.full(shape, 0.707106)
expected_sample = jnp.full(shape, 0.5)*50
expected_rev_f = jnp.full(shape, -5.962035)
expected_rev_g = jnp.full((batch_size,), 2.918423)
expected_rev_f_flow = jnp.full(shape, -2.9810176)


assert jnp.allclose(f, expected_f), 'Mismatch in f'
assert jnp.allclose(g, expected_g), 'Mismatch in g'
assert jnp.allclose(mean, expected_mean), 'Mismatch in mean'
assert jnp.allclose(std, expected_std), 'Mismatch in std'
assert jnp.allclose(prior_sample, expected_sample), 'Mismatch in prior sample'
assert jnp.allclose(rev_f, expected_rev_f), 'Mismatch in rev_f'
assert jnp.allclose(rev_g, expected_rev_g), 'Mismatch in rev_g'
assert jnp.allclose(rev_f_flow, expected_rev_f_flow), 'Mismatch in rev_f when probability flow=True'

## Samplers

### Simple Predictor and Correctors

In [ ]:
shape = (batch_size,4,4,3)
x = jnp.ones(shape)
t = jnp.full((batch_size,),0.5)

In [ ]:
predictor = samplers.Predictor()
corrector = samplers.Corrector()

pred_output = predictor.update_fn(rng=None, x=x, t=None)
corr_output = corrector.update_fn(rng=None, x=x, t=None)

assert jnp.allclose(pred_output[0],x), 'Predictor output does not match expected value'
assert jnp.allclose(pred_output[1],x), 'Predictor mean does not match expected value'
assert jnp.allclose(corr_output[0],x), 'Corrector output does not match expected value'
assert jnp.allclose(corr_output[1],x), 'Corrector mean does not match expected value'

### Dummy Objects and Functions

In [ ]:
class dummySDE:
    def __init__(self):
        self.N = 10
        self.T = 1.
    
    def reverse_sde(self, x,t,score,probability_flow=False):
        f = jnp.full_like(x,2.)
        batch_size=x.shape[0]
        G = jnp.full((batch_size,), 3.)
        return f,G
    
    def prior_sampling(self, rng, shape):
        return jnp.ones(shape)

sde = dummySDE()

In [ ]:
class dummyVPSDE(VPSDE):
    def __init__(self):
        self.N = 10
        self.T = 1.0
        self.discrete_betas = jnp.full((self.N), 0.4)
        self.alphas = jnp.full((self.N), 0.6)
    
    def marginal_prob(self, x, t):
        return 0, 0.3

vp_sde = dummyVPSDE()

In [ ]:
class dummyVESDE(VESDE):
    def __init__(self):
        self.N = 10
        self.T = 1
        self.discrete_sigmas = jnp.array([0.5, 0.3]*5)

    def marginal_prob(self, x, t):
        return 0, 0.3

ve_sde = dummyVESDE()

In [ ]:
def dummy_score_fn(x,t):
    return jnp.full_like(x, 0.7)

### Euler-Maruyama Predictor

$x_{mean} = x-f*dt$

$x_{sample} = x_{mean}+G*\sqrt{dt}*z$

In [ ]:
predictor = samplers.EulerMaruyamaPredictor(sde,dummy_score_fn, False)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = predictor.update_fn(rng,x,t)

expected_mean = jnp.full(shape,0.8)
expected_sample = jnp.full(shape, (16+3*jnp.sqrt(10))/20)

assert jnp.allclose(x_sample, expected_sample, atol=1e-4), "Predictor output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-4), "Predictor mean does not match expected value"

### Reverse Diffusion Predictor

$x_{mean} = x-f$ 

$x_{sample} = x_{mean}+G*z$

In [ ]:
predictor = samplers.ReverseDiffusionPredictor(sde, dummy_score_fn, False)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = predictor.update_fn(rng,x,t)

expected_mean = jnp.full(shape, -1.)
expected_sample = jnp.full(shape, 0.5)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Predictor output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Predictor mean does not match expected value"

### Ancestral Sampling Predictor

#### VP SDE

$x_{mean} = \frac{x+\beta_i * s(x, i)}{\sqrt{1-\beta_i}}$

$x_{sample} = x_{mean} + \sqrt{\beta_i}*z$

In [ ]:
predictor = samplers.AncestralSamplingPredictor(vp_sde, dummy_score_fn, probability_flow=False)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = predictor.update_fn(rng, x, t)

expected_mean = jnp.full(shape, 32*jnp.sqrt(15)/75)
expected_sample = jnp.full(shape, (64*jnp.sqrt(15)+15*jnp.sqrt(10))/150)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Predictor output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Predictor mean does not match expected value"

#### VE SDE

$x_{mean} = x+(\sigma_i^2-\sigma_{i-1}^2)*s(x,i)$

$x_{sample} = x_{mean}+\sqrt{\frac{\sigma_{i-1}^2*(\sigma_i^2-\sigma_{i-1}^2)}{\sigma_i^2}}*z$

In [ ]:
predictor = samplers.AncestralSamplingPredictor(ve_sde, dummy_score_fn, probability_flow=False)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = predictor.update_fn(rng,x,t)

expected_mean = jnp.full(shape, 139/125)
expected_sample = jnp.full(shape, 154/125)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Predictor output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Predictor mean does not match expected value"

### LangevinCorrector

#### VP SDE

In [ ]:
corrector = samplers.LangevinCorrector(vp_sde, dummy_score_fn, 0.1, 2)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = corrector.update_fn(rng, x, t)

expected_mean = jnp.full(shape, 1.0638997)
expected_sample = jnp.full(shape, 1.119228)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Corrector output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Corrector mean does not match expected value"

#### VE SDE

In [ ]:
corrector = samplers.LangevinCorrector(ve_sde, dummy_score_fn, 0.1, 2)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = corrector.update_fn(rng, x, t)

expected_mean = jnp.full(shape, 1.0857143)
expected_sample = jnp.full(shape, 1.1571429)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Corrector output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Corrector mean does not match expected value"

### Annealed Langevin Corrector

#### VP SDE

In [ ]:
corrector = samplers.AnnealedLangevinCorrector(vp_sde, dummy_score_fn, 0.1, 2)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = corrector.update_fn(rng, x, t)

expected_mean = jnp.full(shape, 1.0247499)
expected_sample = jnp.full(shape, 1.0479878)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Corrector output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Corrector mean does not match expected value"

#### VE SDE

In [ ]:
corrector = samplers.AnnealedLangevinCorrector(ve_sde, dummy_score_fn, 0.1, 2)
rng = jax.random.PRNGKey(0)
x_sample, x_mean = corrector.update_fn(rng, x, t)

expected_mean = jnp.full(shape, 1.03252)
expected_sample = jnp.full(shape, 1.06252)

assert jnp.allclose(x_sample, expected_sample, atol=1e-5), "Corrector output does not match expected value"
assert jnp.allclose(x_mean, expected_mean, atol=1e-5), "Corrector mean does not match expected value"

### PC-Sampling

In [ ]:
predictor, corrector = samplers.Predictor(), samplers.Corrector()
inverse_scaler = lambda x: x*2.0

sampler_fn = samplers.pc_sampler(sde, shape, predictor, corrector, inverse_scaler, 1, True)
rng = jax.random.PRNGKey(0)
x_final, evals = sampler_fn(rng)

expected_evals = sde.N*2
expected_x = jnp.full(shape, 2.0)

assert evals == expected_evals, 'Mismatch in number of evaluations'
assert jnp.allclose(x_final, expected_x), "Sampler output does not match expected value"

### ODE-Sampling

In [ ]:
inverse_scaler = lambda x: x*2.0

sampler = samplers.ode_sampler(
    sde=sde,
    score_fn=dummy_score_fn,
    shape=shape,
    inverse_scaler=inverse_scaler,
    denoise=True
)

rng = jax.random.PRNGKey(0)
x_final, nfe = sampler(rng)
assert x_final.shape == shape, "Shape mismatch"
assert nfe > 0, "Zero function evaluations"

## SDEs + Predictors/Correctors

In [ ]:
shape = (batch_size,4,4,3)
x = jnp.ones(shape)
t = jnp.full((batch_size,),0.5)
rng = jax.random.PRNGKey(0)

In [ ]:
vp_sde = VPSDE(0.1, 20, 10)
sub_vp_sde = subVPSDE(0.1, 20, 10)
ve_sde = VESDE(0.01, 50, 10)

### Euler-Maruyama

In [ ]:
em_vp = samplers.EulerMaruyamaPredictor(vp_sde, dummy_score_fn, False)
x_sample, x_mean = em_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
em_sub_vp = samplers.EulerMaruyamaPredictor(sub_vp_sde, dummy_score_fn, False)
x_sample, x_mean = em_sub_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
em_ve = samplers.EulerMaruyamaPredictor(ve_sde, dummy_score_fn, False)
x_sample, x_mean = em_ve.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

### Reverse Diffusion

In [ ]:
rd_vp = samplers.ReverseDiffusionPredictor(vp_sde, dummy_score_fn, False)
x_sample, x_mean = rd_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
rd_sub_vp = samplers.ReverseDiffusionPredictor(sub_vp_sde, dummy_score_fn, False)
x_sample, x_mean = rd_sub_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
rd_ve = samplers.ReverseDiffusionPredictor(ve_sde, dummy_score_fn, False)
x_sample, x_mean = rd_ve.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

### Ancestral Sampling

In [ ]:
as_vp = samplers.AncestralSamplingPredictor(vp_sde, dummy_score_fn, False)
x_sample, x_mean = as_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
as_ve = samplers.AncestralSamplingPredictor(ve_sde, dummy_score_fn, False)
x_sample, x_mean = as_ve.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

### Langevin

In [ ]:
l_vp = samplers.LangevinCorrector(vp_sde, dummy_score_fn, 0.1, 10)
x_sample, x_mean = l_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
l_ve = samplers.LangevinCorrector(ve_sde, dummy_score_fn, 0.1, 10)
x_sample, x_mean = l_ve.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

### Annealed Langevin

In [ ]:
al_vp = samplers.AnnealedLangevinCorrector(vp_sde, dummy_score_fn, 0.1, 10)
x_sample, x_mean = al_vp.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

In [ ]:
al_ve = samplers.AnnealedLangevinCorrector(ve_sde, dummy_score_fn, 0.1, 10)
x_sample, x_mean = al_ve.update_fn(rng, x, t)
assert jnp.all(~jnp.isnan(x_mean)), 'NaN in mean'
assert jnp.all(~jnp.isnan(x_sample)), 'NaN in sample'

### PC Sampling with actual SDEs

In [ ]:
jax.random.normal = normal

In [ ]:
rng = jax.random.PRNGKey(0)
ve_sde = VESDE(0.01, 50, 10)
as_ve = samplers.AncestralSamplingPredictor(ve_sde, dummy_score_fn, False)
al_ve = samplers.AnnealedLangevinCorrector(ve_sde, dummy_score_fn, 0.1, 10)
inverse_scaler = lambda x: x*2.0
sampler = samplers.pc_sampler(ve_sde, shape, as_ve, al_ve, inverse_scaler, n_steps=100, denoise=True)
x_final, evals = sampler(rng)
assert jnp.all(~jnp.isnan(x_final)), 'NaN in sample'
print(x_final)

## Entire System Test

In [ ]:
from config import get_config_local
from train import train
from model_utils import get_score_fn

In [ ]:
configuration = get_config_local()

In [ ]:
params, model, sde = train(configuration)

In [ ]:
from samplers import get_sampler
import jax.numpy as jnp
import jax

In [ ]:
sampler = get_sampler(sde, model, params, configuration)
rng = jax.random.PRNGKey(0)
x_final, _ = sampler(rng)
print(x_final)